# M3 Agentic AI — Email assistant workflow (AISuite)

This notebook recreates the lab you pasted: an **LLM email assistant agent** that uses **tools** to interact with a **simulated email service** (REST API).

> You will need local project files such as `utils.py`, `display_functions.py`, and `email_tools.py` (course-provided), plus the simulated email service running (typically a FastAPI app).


## 0. Inspect your project folder (what files are present?)

Run this cell to see whether the required local files exist next to the notebook.


In [1]:
import os, glob, textwrap

print("📁 Working directory:", os.getcwd())
print("\nFiles here:")
for p in sorted(glob.glob("*")):
    print(" -", p)

required = ["utils.py", "display_functions.py", "email_tools.py"]
missing = [f for f in required if not os.path.exists(f)]
print("\nRequired local files:", required)
if missing:
    print("❌ Missing:", missing)
    print("➡️  Put these files in the same folder as this notebook (or ensure they are on PYTHONPATH).")
else:
    print("✅ All required local files found.")


📁 Working directory: /Users/xmht/Repos/github/border-shepherd/andrew_ng_agentic_course/code/lab/m3/email_assistant_workflow

Files here:
 - M3_Agentic_AI_Email_Assistant_Workflow.ipynb
 - __pycache__
 - display_functions.py
 - email_server.py
 - email_tools.py
 - utils.py

Required local files: ['utils.py', 'display_functions.py', 'email_tools.py']
✅ All required local files found.


## 1. Imports, environment, and AISuite client

- Loads environment variables from `.env` (if present).
- Initializes the AISuite client.
- Imports local helper modules and tool wrappers.


In [2]:
# ================================
# Imports
# ================================

# --- Third-party ---
from dotenv import load_dotenv
import aisuite as ai
import json

# --- Local / project ---
import utils               # local file: utils.py
import display_functions   # local file: display_functions.py
import email_tools         # local file: email_tools.py

# ================================
# Environment & Client
# ================================
load_dotenv()          # Load environment variables from .env
client = ai.Client()   # Initialize AISuite client


## 2. Make sure the simulated email service is running

This lab assumes there is a **local REST service** (often FastAPI) with routes like:

- `POST /send`
- `GET /emails`
- `GET /emails/unread`
- `GET /emails/{id}`
- `GET /emails/search?q=...`
- `PATCH /emails/{id}/read`
- `DELETE /emails/{id}`
- `GET /reset_database`

Usually, `utils.py` and/or `email_tools.py` contains a `BASE_URL` pointing to this server (e.g., `http://127.0.0.1:8000`).

Run the next cell to print any obvious base URL configuration (best-effort; depends on your provided files).


In [3]:
import inspect, re

def find_base_url(module):
    txt = inspect.getsource(module)
    m = re.search(r"(BASE_URL\s*=\s*['\"])([^'\"]+)(['\"])", txt)
    return m.group(2) if m else None

print("utils BASE_URL:", find_base_url(utils))
print("email_tools BASE_URL:", find_base_url(email_tools))


utils BASE_URL: None
email_tools BASE_URL: None


## 3. Sanity-check the backend with `utils.test_*` helpers

Uncomment any lines you want to run. These should hit the simulated service and print results.


In [4]:
# Uncomment the line 'utils.test_*' you want to try

new_email_id = utils.test_send_email()
_ = utils.test_get_email(new_email_id['id'])

_ = utils.test_list_emails()
_ = utils.test_filter_emails(recipient="test@example.com")
_ = utils.test_search_emails("lunch")
_ = utils.test_unread_emails()
_ = utils.test_mark_read(new_email_id['id'])
_ = utils.test_mark_unread(new_email_id['id'])
_ = utils.test_delete_email(new_email_id['id'])
_ = utils.reset_database()


## 4. Tool layer for the email agent

`email_tools.py` should wrap the REST endpoints as Python functions (tools) like:
- `list_all_emails()`
- `list_unread_emails()`
- `search_emails(query)`
- `filter_emails(...)`
- `get_email(email_id)`
- `mark_email_as_read(id)`
- `mark_email_as_unread(id)`
- `send_email(...)`
- `delete_email(id)`
- `search_unread_from_sender(addr)`


In [5]:
# Test sending a new email and fetch it by ID
new_email = email_tools.send_email("test@example.com", "Lunch plans", "Shall we meet at noon?")
content_ = email_tools.get_email(new_email['id'])

# Uncomment the ones you want to try:
# content_ = email_tools.list_all_emails()
# content_ = email_tools.list_unread_emails()
# content_ = email_tools.search_emails("lunch")
# content_ = email_tools.filter_emails(recipient="test@example.com")
# content_ = email_tools.mark_email_as_read(new_email['id'])
# content_ = email_tools.mark_email_as_unread(new_email['id'])
# content_ = email_tools.search_unread_from_sender("test@example.com")
# content_ = email_tools.delete_email(new_email['id'])

utils.print_html(content=json.dumps(content_, indent=2), title="Testing the email_tools")


## 5. Preparing the agent prompt

We wrap a natural-language request with system-style instructions to encourage:
- using tools directly
- no confirmation prompts
- consistent behavior


In [6]:
def build_prompt(request_: str) -> str:
    return f"""
- You are an AI assistant specialized in managing emails.
- You can perform various actions such as listing, searching, filtering, and manipulating emails.
- Use the provided tools to interact with the email system.
- Never ask the user for confirmation before performing an action.
- If needed, my email address is "you@email.com" so you can use it to send emails or perform actions related to my account.

{request_.strip()}
"""

example_prompt = build_prompt("Delete the Happy Hour email")
utils.print_html(content=example_prompt, title="Example prompt")


### 5.3 Resetting the email service

Reset the simulated inbox back to its initial state (useful after experiments).


In [7]:
utils.reset_database()


{'message': 'Database reset to initial state', 'email_count': 5}

## 6. LLM + Email tools

### 6.3 Run a multi-step scenario

Example request:
> “Check for unread emails from boss@email.com, mark them as read, and send a polite follow-up.”


In [8]:
prompt_ = build_prompt("Check for unread emails from boss@email.com, mark them as read, and send a polite follow-up.")

response = client.chat.completions.create(
    model="openai:gpt-4.1",
    messages=[{"role": "user", "content": prompt_}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email
    ],
    max_turns=5,
)

display_functions.pretty_print_chat_completion(response)


## 6.4 Missing tool example: `delete_email` not registered

If a tool isn't provided to the model, it may *want* to do an action but cannot execute it.


In [9]:
prompt_ = build_prompt("Delete alice@work.com email")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": prompt_}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)


## 6.4.1 Try again with `delete_email` enabled

In [10]:
prompt_ = build_prompt("Delete alice@work.com email")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": prompt_}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
        email_tools.delete_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)


## 6.5 Targeted action: Delete the “Happy Hour” email

The simulated inbox typically includes an email with subject **"Happy Hour"**.


In [11]:
prompt_ = build_prompt("Delete the happy hour email")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": prompt_}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
        email_tools.delete_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)


## Appendix: What you need to install / provide

### Standard library (no install)
- `json`

### Third-party packages (pip install)
- `aisuite`
- `python-dotenv`

> Additionally, the **simulated email backend** commonly uses (depending on your course repo):
- `fastapi`
- `uvicorn`
- `sqlalchemy`
- `pydantic`
- `requests` (often used inside `utils.py` / `email_tools.py`)

### Local files you must have (same folder as notebook, or on `PYTHONPATH`)
- `utils.py` (endpoint test helpers + HTML printing utilities)
- `display_functions.py` (pretty print tool-call traces)
- `email_tools.py` (tool wrappers the LLM can call)

### Optional
- `.env` (stores API keys / provider config for AISuite)
